# PaperSloth — Ingestion Pipeline

**Steps covered in this notebook:**
1. Parse PDF with Docling (text + tables + images)
2. Chunk by question boundary
3. Extract & save images, link to chunks
4. Inspect results before moving to LLM enrichment

Next notebook → `02_Enrichment.ipynb` (topic tagging + contextual preamble)

## Step 1 — Parse PDF with Docling

In [1]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from image_handler import get_pipeline_options_with_images

import json
from pathlib import Path

# Use the helper that enables image materialisation
pipeline_options = get_pipeline_options_with_images(num_threads=4)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

PDF_PATH = "./test1.pdf"  
PAPER_ID = "cs301_jan2023" 

CACHE_PATH = Path("./cache/rbb3013_sept2025.json")

if CACHE_PATH.exists():
    # Load from cache — instant
    from docling_core.types.doc import DoclingDocument
    doc = DoclingDocument.model_validate_json(CACHE_PATH.read_text())
    print("Loaded from cache")
else:
    # Parse — slow, only happens once
    result = converter.convert(PDF_PATH)
    doc = result.document
    CACHE_PATH.parent.mkdir(exist_ok=True)
    CACHE_PATH.write_text(doc.model_dump_json())
    print("Parsed and cached")


print(f"Pages:    {len(doc.pages)}")
print(f"Texts:    {len(doc.texts)}")
print(f"Tables:   {len(doc.tables)}")
print(f"Pictures: {len(doc.pictures)}")

Loaded from cache
Pages:    10
Texts:    230
Tables:   4
Pictures: 6


In [3]:
# Add this debug cell and run it BEFORE chunk_questions
print("=== Raw elements on pages 2-7 ===")
for item in doc.texts:
    if hasattr(item, 'prov') and item.prov:
        pg = item.prov[0].page_no
        if 2 <= pg <= 7:
            print(f"p{pg} | {repr(item.text[:80])}")

=== Raw elements on pages 2-7 ===
p2 | 'RBB3013'
p2 | 'a. The output voltage from a precision  12-V power supply is monitored at interv'
p2 | 'TABLEQ1'
p2 | 'Evaluate the average value and standard deviation of the voltage measurement.'
p2 | '[4 marks]'
p2 | 'Analyse  whether  only  random  errors  are  present  during  the measurement. E'
p2 | '[3 marks]'
p2 | 'Justify the important of error analysis for the above production. [3  marks]'
p2 | '2'
p3 | 'RBB3013'
p3 | '_,'
p3 | 'A  multirange  voltmeter  is  constructed  using  a  permanent  magnet  moving  '
p3 | 'SOOOYDC'
p3 | 'DCVollage'
p3 | '22'
p3 | 'W'
p3 | 'R'
p3 | 'R4'
p3 | 'R'
p3 | 'M'
p3 | 'SOV'
p3 | '10V.'
p3 | '250V'
p3 | 'A0COT'
p3 | 'S'
p3 | '2.59'
p3 | 'Raoge selector'
p3 | 'snitch'
p3 | 'SY'
p3 | 'FIGURE Q1b'
p3 | 'Determine the set of series  resistances,  R1, Rz,  Ri and  R4,  required  for t'
p3 | 'L'
p3 | '[10 marks].'
p3 | 'If  the  nleter ist.o b~ fu~her scaled .to  cater for 5..000 V  range,  deterfni'
p3 | '-"h 

## Step 2 — Chunk by question boundary

In [2]:
from question_chunker import chunk_questions

chunks = chunk_questions(doc, split_level="letter")

print(f"Total chunks: {len(chunks)}\n")
for c in chunks:
    print(c)

[chunker] Processing pages 2–7 (skipping cover p1, appendix p8+)
[chunker] WARNING: No question boundaries detected. Check PDF extraction quality or adjust regex patterns.
Total chunks: 1

QuestionChunk(q='1', level='main', marks=10, pages=[2, 3, 4, 5, 6, 7], has_image=True, text='a. The output voltage from a precision  12-V power supply is monitored at interv'...)


In [ ]:
# Inspect a single chunk in detail
IDX = 0  # change this
c = chunks[IDX]
print(f"Question: {c.question_number}")
print(f"Pages:    {c.page_numbers}")
print(f"Marks:    {c.marks}")
print(f"Has img:  {c.has_image}")
print(f"Type:     {c.question_type}")
print("---")
print(c.question_text)

## Step 3 — Extract & link images

In [ ]:
from image_handler import extract_and_link_images

# Save locally (change use_supabase=True when you're ready to deploy)
chunks = extract_and_link_images(
    doc,
    chunks,
    output_dir="./extracted_images",
    paper_id=PAPER_ID,
    use_supabase=False,
)

# Show which chunks have images
img_chunks = [c for c in chunks if c.has_image]
print(f"{len(img_chunks)} chunks have images")
for c in img_chunks:
    print(f"  Q{c.question_number}: {c.image_refs}")

In [ ]:
# Quick visual check — display an extracted image
from PIL import Image
import matplotlib.pyplot as plt

if img_chunks:
    img_path = img_chunks[0].image_refs[0]
    img = Image.open(img_path)
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Q{img_chunks[0].question_number} — extracted image")
    plt.show()
else:
    print("No images found in this paper.")

## Step 4 — Validate before moving on

In [ ]:
# Sanity-check summary — run this to confirm quality before proceeding
import json

print("=== Chunk summary ===")
print(f"Total chunks:         {len(chunks)}")
print(f"With marks detected:  {sum(1 for c in chunks if c.marks)}")
print(f"With images:          {sum(1 for c in chunks if c.has_image)}")
print(f"Empty text (bad!):    {sum(1 for c in chunks if not c.question_text.strip())}")

print("\n=== Type breakdown ===")
from collections import Counter
types = Counter(c.question_type for c in chunks)
for t, count in types.items():
    print(f"  {t}: {count}")

print("\n=== First 5 chunks as JSON ===")
print(json.dumps([c.to_dict() for c in chunks[:5]], indent=2))

## ✅ Done — ready for next step

If the chunk summary looks correct (questions detected, marks parsed, images linked),
proceed to `02_Enrichment.ipynb` for:
- LLM topic tagging
- Contextual preamble generation
- Embedding (Ollama dense + BM25 sparse)

**If chunks look wrong** (too few, splits in wrong places), check:
1. Print `doc.texts` raw to see what Docling extracted
2. Adjust regex patterns in `question_chunker.py` to match your paper's format
3. Try `split_subquestions=False` to only split on main question numbers